# QuantAI specialist model — v4 training and evaluation

This notebook fine-tunes an open-weight instruction model (Qwen3-4B-Instruct) for QuantAI with LoRA/QLoRA. It does **not** call a paid LLM API. The result is a specialised fine-tune of a pretrained model, not a model trained from scratch.

**What v4 changes:** the training and evaluation examples now contain the *exact* context QuantAI's browser sends (dataset variables, data checks, earlier analyses, chat history and retrieved knowledge), built by the same code the local API uses. Every expected action was run in QuantAI's real engine before it was kept. v3 was trained on a short `Context: variables are …` format that the app never sends.

**Before you start:** choose a GPU runtime (Runtime → Change runtime type → T4 GPU). Expected time on a free T4: about 3 hours for the standard run, 5 hours for the full run, plus about 30–45 minutes of evaluation. Checkpoints are saved to Google Drive, so a disconnect only costs the minutes since the last checkpoint.

No Hugging Face token is needed. Do not add secrets or private research data to this notebook.


In [ ]:
import os, subprocess, pathlib, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'QuantAI training needs a GPU runtime. In Colab, choose Runtime > Change runtime type > T4 GPU (if available) > Save, then reconnect and run all cells again. '
        'This notebook cannot turn a GPU on programmatically. Stop here rather than spending time installing packages for a CPU-only training run.'
    )
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', torch.cuda.get_device_properties(0).total_memory // (1024**3), 'GiB')
REPO = '/content/model-analysis--hub'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','quantai-action-completeness-v2','https://github.com/kamaldeenjnr/model-analysis--hub.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin','quantai-action-completeness-v2'], check=True)
    subprocess.run(['git','-C',REPO,'checkout','-B','quantai-action-completeness-v2','FETCH_HEAD'], check=True)
os.chdir(os.path.join(REPO,'quantai-model'))
print('Working directory:', os.getcwd())

In [ ]:
# Remove unrelated preinstalled packages that require Hugging Face Hub 1.x.
# QuantAI's Transformers 4.x training stack uses Hub 0.x; these packages are not used here.
%pip uninstall -y gradio gradio-client diffusers torchao


In [ ]:
# Install only QuantAI's declared training/inference dependencies.
%pip install -q -r requirements.txt


## 0. Keep outputs on Google Drive (recommended)

With `PERSIST_TO_DRIVE = True`, the `outputs` folder lives in `MyDrive/QuantAI/outputs`, so checkpoints, evaluation reports and exported models survive a disconnect. Your v3 files already there are kept: v4 uses its own names. After a disconnect, reconnect and **Run all**: training resumes from the newest checkpoint.


In [ ]:
PERSIST_TO_DRIVE = True  # set False to keep outputs only for this Colab session
import shutil
OUT = pathlib.Path('outputs')
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = pathlib.Path('/content/drive/MyDrive/QuantAI/outputs')
    target.mkdir(parents=True, exist_ok=True)
    if OUT.is_symlink():
        OUT.unlink()
    if OUT.exists():
        # Outputs from earlier in this session: copy them to Drive, keep the local copy aside.
        shutil.copytree(OUT, target, dirs_exist_ok=True)
        OUT.rename('outputs_local_backup')
        print('Copied existing local outputs to Drive; the local copy is in outputs_local_backup/')
    OUT.symlink_to(target, target_is_directory=True)
    print('outputs ->', target)
    print('Already in Drive:', sorted(x.name for x in target.iterdir()) or 'nothing yet')
else:
    OUT.mkdir(exist_ok=True)
    print('Outputs are kept only for this session:', OUT.resolve())


## 1. Validate the data and code

Checks the v4 files (schema, every action against the contract, variables visible in the context, no held-out scenario in training, no near-duplicate wording) and runs the offline tests. These checks do not show that a model is good; the evaluation in section 3 does that.


In [ ]:
!python validate_data.py --train data/v4/train.jsonl --eval data/v4/eval.jsonl
!python datagen/audit.py data/v4/train.jsonl data/v4/eval.jsonl
!pytest -q test_pipeline_v4.py test_local_service.py test_frontend_contract.py


## 2. Train the v4 adapter

Choose the run size:
- **Standard** (`TRAIN_ROWS = 650`): an even sample of every kind of request, about 3 hours on a T4. Fits a free Colab session.
- **Full** (`TRAIN_ROWS = 0`): all 1,124 training rows, about 5 hours. Use it if you have a longer session (or resume after a disconnect).

One epoch is used because each row is a different request; the earlier v3 run needed three epochs over only 93 rows. A checkpoint is written every 10 steps. The first lines of output report the real token lengths; training stops (instead of silently cutting text) if any row is longer than `--max-length`.


In [ ]:
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
TRAIN_ROWS = 650   # 650 = standard (~3 h on a T4); 0 = all 1,124 rows (~5 h)
!python train_lora.py --train data/v4/train.jsonl --eval data/v4/eval.jsonl --output outputs/quantai-qwen3-4b-lora-v4 --epochs 1 --max-train-rows {TRAIN_ROWS} --resume


## 3. Evaluate base, v3 and v4 on the same held-out cases

All models answer the same 236 held-out cases (research scenarios never used for training, worded differently), loaded the same way (4-bit on the GPU):
- **Base**: untouched Qwen3-4B with the v4 instructions — what prompting alone achieves.
- **v3**: your previous adapter, with the v3 prompt it was trained on — how v3 would behave inside the real app.
- **v4**: the new adapter.

Each run takes roughly 10–20 minutes. *Lenient match* accepts only differences that change nothing in the engine (predictor order, `auto` versus the model the engine would pick, swapped variables in a correlation). Field scores show which part of an action goes wrong.


In [ ]:
!python evaluate.py --model outputs/quantai-qwen3-4b-lora-v4 --cases data/v4/eval.jsonl --report outputs/v4-evaluation.json
!python evaluate.py --baseline-only --cases data/v4/eval.jsonl --report outputs/base-evaluation-v4set.json
if pathlib.Path('outputs/quantai-qwen3-4b-lora-v3/adapter_config.json').exists():
    !python evaluate.py --model outputs/quantai-qwen3-4b-lora-v3 --prompt-version v3 --cases data/v4/eval.jsonl --report outputs/v3-evaluation-v4set.json
else:
    print('No v3 adapter in outputs/; skipping the v3 comparison.')


In [ ]:
import json
reports = {name: json.loads(pathlib.Path(f).read_text()) for name, f in
           [('Base', 'outputs/base-evaluation-v4set.json'), ('v3', 'outputs/v3-evaluation-v4set.json'), ('v4', 'outputs/v4-evaluation.json')]
           if pathlib.Path(f).exists()}
rows = [('Valid JSON', 'valid_json_rate'), ('Action contract valid', 'action_contract_valid_rate'), ('Action type correct', 'action_type_accuracy'),
        ('Exact action match', 'exact_action_match_rate'), ('Lenient action match', 'lenient_action_match_rate'),
        ('Declines/asks when it should', 'null_when_expected_rate'), ('Declines when an action was wanted', 'null_when_action_expected_rate'),
        ('Replies with numbers not in context', 'replies_with_numbers_not_in_context_rate')]
print(f"{'Metric':38}" + ''.join(f'{n:>10}' for n in reports))
for label, key in rows:
    print(f'{label:38}' + ''.join(f"{(r.get(key) or 0):10.1%}" for r in reports.values()))
v4 = reports.get('v4')
if v4:
    lo, hi = v4['lenient_action_match_rate_ci95']
    print(f"\nv4 lenient match 95% CI: {lo:.1%} to {hi:.1%} (n = {v4['cases']})")
    print('\nField accuracy (v4):')
    for f, s in v4['field_scores'].items():
        if not f.endswith('precision'):
            print(f"  {f:34} {s['mean']:6.1%}  (n={s['n']})")
    print('\nLenient match by request type:')
    names = list(reports)
    print(f"  {'intent':22}" + ''.join(f'{n:>8}' for n in names) + '     n')
    for intent in v4['by_intent']:
        print(f'  {intent:22}' + ''.join(f"{reports[n]['by_intent'].get(intent, {}).get('lenient_match', 0):8.0%}" for n in names) + f"  {v4['by_intent'][intent]['n']:4}")
    print('\nv4 cases that did not match (first 25):')
    misses = [r for r in v4['results'] if not r['lenient_action_match']]
    for r in misses[:25]:
        got = (r.get('parsed') or {}).get('action') if isinstance(r.get('parsed'), dict) else r.get('raw_output', '')[:120]
        print(f"\n[{r['intent']}] {r['message'][:140]}\n  expected: {json.dumps(r['expected']['action'])[:220]}\n  got:      {json.dumps(got)[:220]}")
    print(f'\n{len(misses)} of {len(v4["results"])} v4 cases did not match.')


## 4. Export a standalone v4 model

Merges the adapter into the base weights (about 8 GB) and records the prompt version, so `serve.py` automatically uses the v4 prompt with this model. Check the base model's licence before redistributing weights.

Optional: the scores above were measured in 4-bit. To confirm the exported fp16 model behaves the same, run the commented line (about 20 minutes).


In [ ]:
!python export_model.py --adapter outputs/quantai-qwen3-4b-lora-v4 --output outputs/quantai-merged-model-v4
print('Export exists:', pathlib.Path('outputs/quantai-merged-model-v4/quantai_prompt.json').exists())
# !python evaluate.py --model outputs/quantai-merged-model-v4 --precision float16 --cases data/v4/eval.jsonl --report outputs/v4-merged-fp16-evaluation.json


## 5. Check what was saved


In [ ]:
for name in ['quantai-qwen3-4b-lora-v4', 'v4-evaluation.json', 'base-evaluation-v4set.json', 'v3-evaluation-v4set.json', 'quantai-merged-model-v4']:
    path = pathlib.Path('outputs') / name
    print(f"{name:32} {'present' if path.exists() else 'MISSING'}")
!du -sh outputs/quantai-merged-model-v4 2>/dev/null


## Release checklist

- [ ] Read the v4 failure list above and decide which failures matter most.
- [ ] Spot-check v4 on a few of your own real questions in the app before relying on it.
- [ ] Have a second person review a sample of training and held-out examples.
- [ ] Serve v4 locally (`QUANTAI_MODEL_PATH=outputs/quantai-merged-model-v4`) and check latency and memory on the target machine.
- [ ] Keep statistical calculations in the deterministic engine and validate every requested action.
- [ ] Do not claim production readiness until independent review and acceptance thresholds pass.
